# Пункт 6. Содержательный SFT-пилот v3

**Новая сессия Colab · T4 GPU · выполнять сверху вниз.**

Обучаем свежую LoRA на исходной `Qwen/Qwen3-4B-Instruct-2507`:
80 train-примеров, одна эпоха, LR=5e-5, 20 шагов оптимизатора.
На 16 отдельных validation-примерах измеряем loss.

На 32 development-вопросах сохраняем 32 ответа исходной модели
и 32 ответа после обучения. Для каждой пары совпадают system prompt,
параметры sampling и seed. Обучение начинается после сохранения всех
исходных ответов. Адаптер из smoke / tiny загружать не нужно.

В конце скачивается **один ZIP**: веса, метрики, все ответы и отдельный
`judge/for_judge.zip` с 64 заданиями судье. Вызов внешнего судьи выполняется
отдельно. `PILOT PIPELINE PASS` подтверждает техническую готовность результата.
Улучшение качества оцениваем по содержанию ответов и оценкам судьи.

[Подробная инструкция](https://github.com/maria-igntv/lm-hw/blob/main/docs/sft_v3_pilot_guide.md).


In [ ]:
import importlib.metadata
import json
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path('/content/lm-hw-v3')
if not ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/maria-igntv/lm-hw.git', str(ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(ROOT), 'pull', '--ff-only'], check=True)
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
settings = json.loads((ROOT / 'data/sft_v3/pilot_config.json').read_text(encoding='utf-8'))
selection = json.loads((ROOT / settings['selection_path']).read_text(encoding='utf-8'))
requirements = [f'{package}=={version}' for package, version in settings['packages'].items()]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *requirements], check=True)
for package, version in settings['packages'].items():
    loaded = sys.modules.get(package.replace('-', '_'))
    if loaded is not None and getattr(loaded, '__version__', version) != version:
        raise RuntimeError('Импортирована другая версия библиотеки. Перезапустите Python и выполните ячейки заново.')
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
print('Модель', selection['model_id'])
print('Ревизия', selection['revision'])
print('Train 80, validation 16, development 32, optimizer steps', settings['optimizer_steps'])


## 1. Проверяем данные и фиксируем условия

Train и validation разделены по группам сценариев. Development уже
использовался при выборе модели; это набор для разработки.
Для финального вывода позже понадобится отдельный тест.
Сохраняем входы, хеши, код и полные версии библиотек.


In [ ]:
import platform
import shutil
import time
import traceback
from collections import Counter
from datetime import datetime, timezone

import torch
from checks.screening_v3 import read_jsonl, sha256_file, write_json, write_jsonl, finish_reason
from checks.sft_smoke_v3 import build_masked_example
from checks.sft_tiny_v3 import evaluate_model, train_tiny_model as train_lora
from checks.sft_pilot_v3 import (
    select_pilot_records, generate_pilot_phase, validate_pilot_records, export_pilot_artifacts,
)
from script.prepare_sft_v3 import normalize

if not torch.cuda.is_available():
    raise RuntimeError('Выберите Среда выполнения → Сменить среду выполнения → T4 GPU.')
if 'pilot_model' in globals() or 'OUT_DIR' in globals():
    raise RuntimeError('Для нового полного запуска создайте новую GPU-сессию. Текущий результат можно скачать последней ячейкой.')
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False
CORPUS_DIR = ROOT / settings['corpus_directory']
DEV_DIR = ROOT / settings['development_directory']
for directory, expected in [
    (CORPUS_DIR, settings['corpus_manifest_sha256']),
    (DEV_DIR, settings['development_manifest_sha256']),
]:
    if sha256_file(directory / 'manifest.json') != expected:
        raise ValueError('Версия данных отличается от зафиксированной: ' + str(directory))
    subprocess.run([sys.executable, str(directory / 'validate.py')], check=True)
corpus_manifest = json.loads((CORPUS_DIR / 'manifest.json').read_text(encoding='utf-8'))
dev_manifest = json.loads((DEV_DIR / 'manifest.json').read_text(encoding='utf-8'))
train_rows, validation_rows = select_pilot_records(
    read_jsonl(CORPUS_DIR / 'records.jsonl'), selection['system_prompt'],
)
questions = read_jsonl(DEV_DIR / 'questions.jsonl')
rubrics = read_jsonl(DEV_DIR / 'rubric.jsonl')
sources = json.loads((DEV_DIR / 'sources.json').read_text(encoding='utf-8'))['sources']
review_notes = json.loads((ROOT / 'judge/model_screening_v3/review_notes.json').read_text(encoding='utf-8'))
if len(questions) != 32:
    raise ValueError('Ожидались 32 development-вопроса')
if {normalize(row['messages'][1]['content']) for row in train_rows + validation_rows} & {
    normalize(row['question']) for row in questions
}:
    raise ValueError('Точный повтор вопроса между корпусом и development')
if len(train_rows) != settings['optimizer_steps'] * settings['gradient_accumulation_steps']:
    raise ValueError('Число шагов не соответствует одной эпохе')

RUN_ID = 'qwen3_4b_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f')
OUT_DIR = ROOT / 'results/sft_v3_pilot' / RUN_ID
OUT_DIR.mkdir(parents=True, exist_ok=False)
ADAPTER_DIR = OUT_DIR / 'adapter'
tracked_files = [
    'data/sft_v3/selection.json', 'data/sft_v3/pilot_config.json',
    'checks/sft_smoke_v3.py', 'checks/sft_tiny_v3.py', 'checks/sft_pilot_v3.py',
    'checks/screening_v3.py', 'checks/autocheck.py', 'checks/evaluation.py',
    'script/prepare_sft_v3.py', 'notebooks/04_sft_v3_pilot.ipynb',
    'data/sft_v3/corpus_v1/manifest.json', 'data/sft_v3/corpus_v1/validate.py',
    'data/development_v3/manifest.json', 'judge/judge_prompt_v3.md',
    'judge/judge_response_schema_v3.json', 'judge/model_screening_v3/review_notes.json',
]
tracked_files.extend(str((CORPUS_DIR / name).relative_to(ROOT)) for name in corpus_manifest['data_file_sha256'])
tracked_files.extend(str((DEV_DIR / name).relative_to(ROOT)) for name in dev_manifest['file_sha256'])
tracked_files = sorted(set(tracked_files))
metadata = {
    'run_id': RUN_ID, 'status': 'prepared', 'experiment_id': settings['experiment_id'],
    'started_at_utc': datetime.now(timezone.utc).isoformat(), 'selection': selection, 'settings': settings,
    'repo_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip(),
    'file_sha256': {name: sha256_file(ROOT / name) for name in tracked_files},
    'versions': {package: importlib.metadata.version(package) for package in ['torch', *settings['packages']]},
    'hardware': {
        'gpu': torch.cuda.get_device_name(0), 'python': platform.python_version(), 'torch_cuda': torch.version.cuda,
        'gpu_memory_gib': torch.cuda.get_device_properties(0).total_memory / 1024**3,
        'nvidia_smi': subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv,noheader'], text=True,
        ).strip(),
    },
    'generation_context': 'Baseline and SFT use the same prepared NF4 base; fresh zero-contribution LoRA verified before baseline.',
    'corpus_review_status': corpus_manifest['status'],
    'quality_status': 'awaiting_judge_scores_and_content_review',
}
write_json(OUT_DIR / 'run_metadata.json', metadata)
write_jsonl(OUT_DIR / 'errors.jsonl', [])
write_jsonl(OUT_DIR / 'training_subset.jsonl', train_rows)
write_jsonl(OUT_DIR / 'validation_subset.jsonl', validation_rows)
for name in tracked_files:
    destination = OUT_DIR / 'reproduction' / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ROOT / name, destination)
(OUT_DIR / 'pip_freeze.txt').write_text(
    subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True), encoding='utf-8',
)
training_state = {'started': False, 'completed': False}


def record_failure(stage, error):
    errors = read_jsonl(OUT_DIR / 'errors.jsonl')
    errors.append({'stage': stage, 'type': type(error).__name__, 'message': str(error), 'traceback': traceback.format_exc()})
    write_jsonl(OUT_DIR / 'errors.jsonl', errors)
    metadata['status'] = 'failed'
    metadata['failed_stage'] = stage
    write_json(OUT_DIR / 'run_metadata.json', metadata)


print(json.dumps(metadata['hardware'], ensure_ascii=False, indent=2))
print('Train', len(train_rows), 'Validation', len(validation_rows), 'Development', len(questions))
print('Run ID', RUN_ID)
print('Результаты', OUT_DIR)


## 2. Загружаем базу и свежую LoRA

Фиксированы ревизия модели, NF4, FP16, eager attention, r16 / alpha32.
Системный промпт, пользовательский вопрос и padding исключены из loss.
Ответ ассистента и EOS включены. Усечение примеров запрещено.

Подготовка к k-bit может немного изменить logits. Это измеряется отдельно.
Baseline генерируется после подготовки, чтобы состояние базовых весов
совпадало с состоянием при оценке SFT.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, GenerationConfig, set_seed
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from safetensors import safe_open

if 'pilot_model' in globals():
    raise RuntimeError('Модель уже загружена. Продолжайте со следующей ячейки или создайте новую сессию.')
set_seed(settings['seed'])
try:
    load_started = time.perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(selection['model_id'], revision=selection['revision'], trust_remote_code=False)
    tokenizer.padding_side = 'right'
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    quantization = dict(settings['quantization'])
    quantization['bnb_4bit_compute_dtype'] = torch.float16
    pilot_model = AutoModelForCausalLM.from_pretrained(
        selection['model_id'], revision=selection['revision'], trust_remote_code=False,
        dtype=torch.float16, device_map={'': 0}, quantization_config=BitsAndBytesConfig(**quantization),
        attn_implementation=settings['attention'],
    ).eval()
    if pilot_model.__class__.__name__ != 'Qwen3ForCausalLM':
        raise ValueError('Неожиданная архитектура выбранной базовой модели')
    original_generation = pilot_model.generation_config.to_dict()
    eos_ids = original_generation['eos_token_id']
    if isinstance(eos_ids, int):
        eos_ids = [eos_ids]
    if set(eos_ids) != {151645, 151643}:
        raise ValueError('Неожиданные EOS выбранной Qwen-модели')
    if not getattr(pilot_model, 'is_loaded_in_4bit', False):
        raise ValueError('Не подтверждена загрузка NF4')
    if any(parameter.device.type != 'cuda' for parameter in pilot_model.parameters()):
        raise ValueError('Часть весов оказалась вне GPU')
    generation_config = GenerationConfig(**settings['generation'], eos_token_id=eos_ids, pad_token_id=tokenizer.pad_token_id)
    control_config = GenerationConfig(
        do_sample=False, max_new_tokens=settings['control_max_new_tokens'],
        eos_token_id=eos_ids, pad_token_id=tokenizer.pad_token_id, use_cache=True,
    )
    train_pairs = [build_masked_example(tokenizer, row, settings['max_seq_length']) for row in train_rows]
    validation_pairs = [build_masked_example(tokenizer, row, settings['max_seq_length']) for row in validation_rows]
    train_batches, train_audit = map(list, zip(*train_pairs))
    validation_batches, validation_audit = map(list, zip(*validation_pairs))
    write_json(OUT_DIR / 'mask_audit.json', {'train': train_audit, 'validation': validation_audit})
    question_index = {row['id']: row for row in questions}
    control_ids = settings['control_question_ids']

    def prompt_inputs(question, system_prompt):
        return tokenizer.apply_chat_template(
            [{'role': 'system', 'content': system_prompt}, {'role': 'user', 'content': question}],
            add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors='pt',
        ).to('cuda')

    @torch.inference_mode()
    def collect_logits():
        pilot_model.eval()
        result = {}
        for question_id in control_ids:
            inputs = prompt_inputs(question_index[question_id]['question'], selection['system_prompt'])
            logits = pilot_model(**inputs, use_cache=False, logits_to_keep=1).logits[0, -1].float().cpu()
            if not torch.isfinite(logits).all():
                raise FloatingPointError('Контрольные logits содержат NaN / Inf')
            result[question_id] = logits
        return result

    @torch.inference_mode()
    def generate_controls():
        pilot_model.eval()
        result = {}
        for question_id in control_ids:
            inputs = prompt_inputs(question_index[question_id]['question'], selection['system_prompt'])
            output = pilot_model.generate(**inputs, generation_config=control_config)
            result[question_id] = output[0, inputs['input_ids'].shape[1]:].tolist()
        return result

    original_logits = collect_logits()
    pilot_model = prepare_model_for_kbit_training(
        pilot_model, use_gradient_checkpointing=settings['gradient_checkpointing'],
        gradient_checkpointing_kwargs={'use_reentrant': settings['use_reentrant']},
    )
    pilot_model.config.use_cache = False
    prepared_logits = collect_logits()
    pilot_model = get_peft_model(pilot_model, LoraConfig(**settings['lora']))
    active_logits = collect_logits()
    with pilot_model.disable_adapter():
        fresh_disabled_logits = collect_logits()
    b_parameters = [parameter for name, parameter in pilot_model.named_parameters() if 'lora_B' in name]
    trainable_tensor_count = sum(parameter.requires_grad for parameter in pilot_model.parameters())
    identity = {
        'B_matrices': len(b_parameters), 'trainable_tensors': trainable_tensor_count,
        'all_B_zero': bool(b_parameters) and all(torch.count_nonzero(parameter).item() == 0 for parameter in b_parameters),
        'active_vs_disabled_allclose': all(torch.allclose(active_logits[key], fresh_disabled_logits[key], rtol=1e-5, atol=1e-5) for key in control_ids),
        'active_vs_disabled_max_abs_diff': max(float((active_logits[key] - fresh_disabled_logits[key]).abs().max()) for key in control_ids),
        'preparation_max_abs_diff': max(float((original_logits[key] - prepared_logits[key]).abs().max()) for key in control_ids),
        'base_weights_frozen': all(not parameter.requires_grad for name, parameter in pilot_model.named_parameters() if 'lora_' not in name),
        'gradient_checkpointing': pilot_model.is_gradient_checkpointing,
        'trainable_parameters': sum(parameter.numel() for parameter in pilot_model.parameters() if parameter.requires_grad),
        'control_question_ids': control_ids,
    }
    write_json(OUT_DIR / 'fresh_adapter_check.json', identity)
    if not all(identity[key] for key in ['all_B_zero', 'active_vs_disabled_allclose', 'base_weights_frozen', 'gradient_checkpointing']):
        raise ValueError('Не прошла проверка свежей LoRA')
    del b_parameters
    metadata.update({
        'model_load_seconds': time.perf_counter() - load_started, 'status': 'fresh_adapter_verified',
        'generation_config': generation_config.to_dict(), 'control_generation_config': control_config.to_dict(),
    })
    write_json(OUT_DIR / 'run_metadata.json', metadata)
except (Exception, KeyboardInterrupt) as error:
    record_failure('load_masks_and_fresh_adapter', error)
    raise

print(json.dumps(identity, ensure_ascii=False, indent=2))


## 3. Сохраняем исходные показатели и 32 baseline-ответа

Вход генерации состоит только из system и user. Эталон, критерии и
категория не подаются модели. Sampling совпадает с development-скринингом:
temperature 0,7, top_p 0,8, top_k 20, лимит 768 токенов.

Каждый ответ сохраняется сразу. При ошибке генерации повторите эту ячейку
в текущей сессии: сохранённые ответы будут пропущены.
После начала обучения запуск baseline заблокирован.


In [ ]:
def save_evaluation(step):
    evaluations = read_jsonl(OUT_DIR / 'evaluations.jsonl')
    for split, batches, rows in [('train', train_batches, train_rows), ('validation', validation_batches, validation_rows)]:
        result = evaluate_model(pilot_model, batches, [row['sample_id'] for row in rows], 'cuda', torch.float16)
        evaluations = [row for row in evaluations if (row['step'], row['split']) != (step, split)]
        evaluations.append({'step': step, 'split': split, **result})
        write_jsonl(OUT_DIR / 'evaluations.jsonl', evaluations)
        print('step', step, split, 'loss', round(result['token_weighted_loss'], 4),
              'token accuracy', round(result['token_accuracy'], 4), 'EOS accuracy', round(result['eos_accuracy'], 4))


@torch.inference_mode()
def generate_one(question, system_prompt, seed):
    pilot_model.eval()
    inputs = prompt_inputs(question, system_prompt)
    input_count = inputs['input_ids'].shape[1]
    if input_count + settings['generation']['max_new_tokens'] > pilot_model.config.max_position_embeddings:
        raise ValueError('Контекст генерации превышает лимит модели')
    set_seed(seed)
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    output = pilot_model.generate(**inputs, generation_config=generation_config)
    torch.cuda.synchronize()
    token_ids = output[0, input_count:].tolist()
    answer = tokenizer.decode(token_ids, skip_special_tokens=True)
    result = {
        'answer': answer, 'raw_answer': tokenizer.decode(token_ids, skip_special_tokens=False),
        'generated_token_ids': token_ids, 'generated_tokens': len(token_ids),
        'eos_token_ids': eos_ids, 'max_new_tokens': settings['generation']['max_new_tokens'],
        'finish_reason': finish_reason(token_ids, eos_ids, settings['generation']['max_new_tokens']),
        'elapsed_seconds': time.perf_counter() - started,
        'peak_allocated_gib': torch.cuda.max_memory_allocated() / 1024**3,
        'peak_reserved_gib': torch.cuda.max_memory_reserved() / 1024**3,
        'input_tokens': input_count, 'input_token_ids': inputs['input_ids'][0].tolist(),
        'rendered_prompt': tokenizer.decode(inputs['input_ids'][0].tolist(), skip_special_tokens=False),
    }
    return result


def on_answer_saved(row, count):
    metadata['saved_answers'] = count
    write_json(OUT_DIR / 'run_metadata.json', metadata)
    print(count, '/ 64', row['phase'], row['id'], 'tokens', row['generated_tokens'], row['finish_reason'])
    print(row['answer'][:300])


if training_state['started']:
    raise RuntimeError('Обучение уже начиналось. Исходные ответы сохранены в answers.jsonl. Продолжайте после обучения.')
try:
    existing_evaluations = {(row['step'], row['split']) for row in read_jsonl(OUT_DIR / 'evaluations.jsonl')}
    if not {(0, 'train'), (0, 'validation')} <= existing_evaluations:
        save_evaluation(0)
    generate_pilot_phase(
        OUT_DIR / 'answers.jsonl', 'baseline', questions, settings, selection, RUN_ID, generate_one, on_answer_saved,
    )
    metadata['status'] = 'baseline_saved'
    write_json(OUT_DIR / 'run_metadata.json', metadata)
except (Exception, KeyboardInterrupt) as error:
    record_failure('baseline', error)
    raise


## 4. Одна эпоха на 80 train-примерах

20 шагов по 4 микробатча. Каждый train-пример используется ровно один раз.
Validation в градиенты не входит. Train и validation измеряются на шагах
0, 10 и 20. На всех шагах LR=5e-5; итоговый checkpoint — шаг 20.

Сразу после обучения сохраняются веса адаптера. Повторно обучать эту LoRA
в той же сессии нельзя. При частичной ошибке обучения скачайте результат
последней ячейкой и сохраните IPYNB с выводом.


In [ ]:
def save_adapter():
    ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    pilot_model.peft_config['default'].base_model_name_or_path = selection['model_id']
    pilot_model.peft_config['default'].revision = selection['revision']
    pilot_model.save_pretrained(ADAPTER_DIR, safe_serialization=True, save_embedding_layers=False)
    tokenizer.save_pretrained(ADAPTER_DIR)
    weights_path = ADAPTER_DIR / 'adapter_model.safetensors'
    tensors = []
    with safe_open(weights_path, framework='pt', device='cpu') as saved:
        for key in saved.keys():
            tensor = saved.get_tensor(key)
            if not torch.isfinite(tensor).all():
                raise FloatingPointError('Сохранённые веса содержат NaN / Inf')
            tensors.append({'name': key, 'shape': list(tensor.shape), 'dtype': str(tensor.dtype)})
    if len(tensors) != identity['trainable_tensors']:
        raise ValueError('Число сохранённых LoRA-тензоров отличается от ожидаемого')
    audit = {'adapter_sha256': sha256_file(weights_path), 'adapter_bytes': weights_path.stat().st_size,
             'saved_tensors': len(tensors), 'all_finite': True, 'tensors': tensors}
    write_json(OUT_DIR / 'adapter_audit.json', audit)
    metadata['adapter_sha256'] = audit['adapter_sha256']
    write_json(OUT_DIR / 'run_metadata.json', metadata)
    return audit


if training_state['started']:
    raise RuntimeError('Обучение уже запускалось. При успехе продолжайте к SFT-ответам; при ошибке сохраните ZIP.')
if metadata['status'] != 'baseline_saved':
    raise RuntimeError('Сначала завершите все 32 baseline-ответа и исходные измерения')
training_state['started'] = True
training_history = []


def on_step(entry):
    training_history.append(entry)
    write_jsonl(OUT_DIR / 'training_steps.jsonl', training_history)
    metadata['last_optimizer_step'] = entry['step']
    write_json(OUT_DIR / 'run_metadata.json', metadata)
    print('step', entry['step'], '/', settings['optimizer_steps'], 'loss', round(entry['token_weighted_loss'], 4),
          'grad norm', round(entry['gradient_norm_before_clip'], 4))


try:
    metadata['status'] = 'training'
    write_json(OUT_DIR / 'run_metadata.json', metadata)
    set_seed(settings['seed'])
    torch.cuda.empty_cache()
    training_summary = train_lora(
        pilot_model, train_batches, [row['sample_id'] for row in train_rows], settings,
        'cuda', torch.float16, on_step, save_evaluation,
    )
    training_summary['epochs_completed'] = training_summary.pop('equivalent_passes_over_tiny_set')
    seen_ids = [item['sample_id'] for step in training_history for item in step['microbatches']]
    training_summary['each_train_example_used_once'] = Counter(seen_ids) == Counter(row['sample_id'] for row in train_rows)
    write_json(OUT_DIR / 'training_summary.json', training_summary)
    if (training_summary['optimizer_steps'] != settings['optimizer_steps']
            or training_summary['epochs_completed'] != 1
            or not training_summary['each_train_example_used_once']
            or training_summary['nonzero_B_matrices'] != identity['B_matrices']):
        raise ValueError('Одна эпоха или обновление всех LoRA B-матриц не подтверждены')
    adapter_audit = save_adapter()
    training_state['completed'] = True
    metadata['status'] = 'training_complete'
    write_json(OUT_DIR / 'run_metadata.json', metadata)
except (Exception, KeyboardInterrupt) as error:
    record_failure('training', error)
    raise

print(json.dumps(training_summary, ensure_ascii=False, indent=2))
print('Веса сохранены', ADAPTER_DIR)


## 5. Генерируем 32 ответа SFT и проверяем базу

Те же вопросы, промпт, sampling и seed каждого вопроса.
Записи дополняют answers.jsonl до 64 строк. Повтор ячейки после ошибки
генерации продолжает незавершённую серию.

На трёх контрольных вопросах проверяем вклад обученной LoRA и совпадение
logits отключённой базы с её исходным подготовленным состоянием.


In [ ]:
if not training_state['completed']:
    raise RuntimeError('Обучение ещё не завершено')
try:
    answers = generate_pilot_phase(
        OUT_DIR / 'answers.jsonl', 'sft', questions, settings, selection, RUN_ID, generate_one, on_answer_saved,
    )
    validate_pilot_records(questions, answers, settings, selection, RUN_ID, complete=True)
    trained_logits = collect_logits()
    with pilot_model.disable_adapter():
        trained_disabled_logits = collect_logits()
    influence = {
        'control_question_ids': control_ids,
        'trained_adapter_max_abs_diff': max(float((trained_logits[key] - trained_disabled_logits[key]).abs().max()) for key in control_ids),
        'disabled_base_max_abs_diff': max(float((trained_disabled_logits[key] - fresh_disabled_logits[key]).abs().max()) for key in control_ids),
        'disabled_base_allclose': all(torch.allclose(trained_disabled_logits[key], fresh_disabled_logits[key], rtol=1e-5, atol=1e-5) for key in control_ids),
    }
    write_json(OUT_DIR / 'trained_adapter_check.json', influence)
    if influence['trained_adapter_max_abs_diff'] == 0 or not influence['disabled_base_allclose']:
        raise ValueError('Не прошла проверка обученного адаптера или отключённой базы')
    metadata['status'] = 'sft_answers_saved'
    write_json(OUT_DIR / 'run_metadata.json', metadata)
except (Exception, KeyboardInterrupt) as error:
    record_failure('sft_generation', error)
    raise

print(json.dumps(influence, ensure_ascii=False, indent=2))


## 6. Проверяем загрузку сохранённого адаптера

Три фиксированных вопроса: dev3_01, dev3_07, dev3_25.
До и после загрузки сравниваем logits первого токена и greedy-последовательности
с лимитом 64 токена. Это дополнительный технический контроль.
Основные 64 ответа для судьи сохраняют sampling и лимит 768.


In [ ]:
try:
    if not training_state['completed']:
        raise RuntimeError('Обучение не завершено')
    validate_pilot_records(questions, read_jsonl(OUT_DIR / 'answers.jsonl'), settings, selection, RUN_ID, complete=True)
    adapter_audit = save_adapter()
    before_reload_logits = collect_logits()
    before_reload_tokens = generate_controls()
    base_for_reload = pilot_model.unload()
    del pilot_model
    pilot_model = PeftModel.from_pretrained(base_for_reload, ADAPTER_DIR, is_trainable=False).eval()
    pilot_model.config.use_cache = False
    reloaded_logits = collect_logits()
    reloaded_tokens = generate_controls()
    per_question = [{
        'question_id': key,
        'logits_max_abs_diff': float((before_reload_logits[key] - reloaded_logits[key]).abs().max()),
        'logits_allclose': torch.allclose(before_reload_logits[key], reloaded_logits[key], rtol=1e-4, atol=1e-4),
        'greedy_tokens_equal': before_reload_tokens[key] == reloaded_tokens[key],
        'before_reload_tokens': before_reload_tokens[key], 'after_reload_tokens': reloaded_tokens[key],
    } for key in control_ids]
    roundtrip = {
        'adapter_sha256': adapter_audit['adapter_sha256'], 'per_question': per_question,
        'generation': control_config.to_dict(),
        'all_logits_match': all(row['logits_allclose'] for row in per_question),
        'all_greedy_tokens_match': all(row['greedy_tokens_equal'] for row in per_question),
    }
    write_json(OUT_DIR / 'save_reload_check.json', roundtrip)
    if not roundtrip['all_logits_match'] or not roundtrip['all_greedy_tokens_match']:
        raise ValueError('Контрольные результаты после загрузки адаптера отличаются')
    metadata['status'] = 'reload_verified'
    write_json(OUT_DIR / 'run_metadata.json', metadata)
except (Exception, KeyboardInterrupt) as error:
    record_failure('save_and_reload', error)
    raise

print('Загрузка проверена на', len(per_question), 'вопросах')
print('SHA-256 адаптера', roundtrip['adapter_sha256'])


## 7. Собираем сравнение и 64 задания судье

В judge/for_judge.zip входят запросы в случайном порядке без названия
модели и без указания baseline/SFT. Текст ответа передаётся точно как
сохранён. Пустые и обрезанные ответы также включаются в оценку.
Служебная таблица соответствия judge_mapping.json хранится отдельно.

Далее пакет можно передать тому же Qwen3.8-27B, при возможности — также
Gemma. Внешняя оценка по рубрике v3 выполняется после этого ноутбука.


In [ ]:
try:
    can_retry_export = metadata.get('failed_stage') == 'summary_and_judge_export'
    if metadata['status'] not in {'reload_verified', 'complete'} and not can_retry_export:
        raise RuntimeError('Сначала завершите проверку загрузки адаптера')
    answers = read_jsonl(OUT_DIR / 'answers.jsonl')
    count = export_pilot_artifacts(
        OUT_DIR, questions, answers, rubrics, sources, settings, selection, RUN_ID,
        (ROOT / 'judge/judge_prompt_v3.md').read_text(encoding='utf-8'),
        json.loads((ROOT / 'judge/judge_response_schema_v3.json').read_text(encoding='utf-8')),
        review_notes,
    )
    evaluations = read_jsonl(OUT_DIR / 'evaluations.jsonl')
    evaluation_index = {(row['step'], row['split']): row for row in evaluations}
    expected_evaluations = {(step, split) for step in settings['evaluation_steps'] for split in ['train', 'validation']}
    if set(evaluation_index) != expected_evaluations:
        raise ValueError('Не все измерения train / validation сохранены')
    fit_summary = {'by_split': {}, 'checkpoint_step': settings['optimizer_steps'],
                   'quality_status': 'awaiting_judge_scores_and_content_review'}
    for split in ['train', 'validation']:
        before = evaluation_index[0, split]
        after = evaluation_index[settings['optimizer_steps'], split]
        fit_summary['by_split'][split] = {
            'n': before['n'], 'loss_before': before['token_weighted_loss'], 'loss_after': after['token_weighted_loss'],
            'loss_decreased': after['token_weighted_loss'] < before['token_weighted_loss'],
            'token_accuracy_before': before['token_accuracy'], 'token_accuracy_after': after['token_accuracy'],
            'eos_accuracy_before': before['eos_accuracy'], 'eos_accuracy_after': after['eos_accuracy'],
        }
    write_json(OUT_DIR / 'fit_summary.json', fit_summary)
    automatic_summary = json.loads((OUT_DIR / 'automatic_summary.json').read_text(encoding='utf-8'))
    if count != 64 or not automatic_summary['complete']:
        raise ValueError('Ожидались 64 ответа и 64 задания судье')
    checks = {
        'status': 'PASS', 'train_examples': len(train_rows), 'validation_examples': len(validation_rows),
        'development_questions': len(questions), 'saved_answers': len(answers), 'judge_requests': count,
        'fresh_adapter_identity': identity['active_vs_disabled_allclose'],
        'base_weights_frozen': identity['base_weights_frozen'], 'optimizer_steps': training_summary['optimizer_steps'],
        'each_train_example_used_once': training_summary['each_train_example_used_once'],
        'trained_adapter_changes_logits': influence['trained_adapter_max_abs_diff'] > 0,
        'disabled_base_controls_match': influence['disabled_base_allclose'],
        'saved_weights_finite': adapter_audit['all_finite'], 'reload_logits_match': roundtrip['all_logits_match'],
        'reload_greedy_tokens_match': roundtrip['all_greedy_tokens_match'],
        'reload_control_question_ids': control_ids, 'logged_errors': len(read_jsonl(OUT_DIR / 'errors.jsonl')),
        'quality_status': 'awaiting_judge_scores_and_content_review',
    }
    write_json(OUT_DIR / 'checks.json', checks)
    metadata['status'] = 'complete'
    metadata.pop('failed_stage', None)
    metadata['finished_at_utc'] = datetime.now(timezone.utc).isoformat()
    write_json(OUT_DIR / 'run_metadata.json', metadata)
except (Exception, KeyboardInterrupt) as error:
    record_failure('summary_and_judge_export', error)
    raise

print('PILOT PIPELINE PASS')
print(json.dumps(fit_summary, ensure_ascii=False, indent=2))
print(json.dumps(automatic_summary['by_prompt'], ensure_ascii=False, indent=2))
print('Для судьи', OUT_DIR / 'judge/for_judge.zip')
print('Качество: ожидаются оценки судьи и просмотр comparison.md')


## 8. Скачиваем ZIP и выполненный ноутбук

Ячейка запускает скачивание через браузер, обычно в «Загрузки» компьютера.
Один ZIP содержит всю папку запуска, включая adapter/ и judge/for_judge.zip.
Её можно выполнить после ошибки, если OUT_DIR уже создан.

Сохраните выполненный IPYNB: **Файл → Скачать → Скачать файл IPYNB**.
Пришлите ZIP и IPYNB одного запуска для анализа. Отдельно пакет
judge/for_judge.zip можно передать на разметку Qwen3.8-27B.


In [ ]:
from google.colab import files

archive = shutil.make_archive(
    str(Path('/content') / ('sft_v3_pilot_' + RUN_ID)),
    'zip', root_dir=OUT_DIR.parent, base_dir=OUT_DIR.name,
)
files.download(archive)
print('Скачивается', archive)
print('Run ID', RUN_ID)
print('Статус', json.loads((OUT_DIR / 'run_metadata.json').read_text(encoding='utf-8'))['status'])
